In [4]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# tscv
from sklearn.model_selection import TimeSeriesSplit

# %%
# Define configuration for the three additional targets
targets_config = {
    "Wind_kW": {
        "exog_cols": ["wind_speed", "wind_speed_100m"],
        "clip_zero": True,
        "forecast_col": "Wind_Forecast_kW",
    },
    "Demand_kW": {
        "exog_cols": ["temp_air"],
        "clip_zero": True,
        "forecast_col": "Demand_Forecast_kW",
    },
    "Price_EUR_per_kWh": {
        "exog_cols": [
            "temp_air",
            "wind_speed",
            "ghi",
        ],  # weather indicators that drive pricing via grid supply/demand
        "clip_zero": False,  # wholesale electricity prices can physically be negative
        "forecast_col": "Price_Forecast_EUR_per_kWh",
    },
}

# Start with a clean copy of the physical dataframe
df_feat_multi = pd.read_csv("synthetic_df_train.csv", index_col=0, parse_dates=True)

# Ensure calendar features are calculated
df_feat_multi["Hour"] = df_feat_multi.index.hour
df_feat_multi["Hour_Sin"] = np.sin(2 * np.pi * df_feat_multi["Hour"] / 24)
df_feat_multi["Hour_Cos"] = np.cos(2 * np.pi * df_feat_multi["Hour"] / 24)
df_feat_multi["DayOfYear"] = df_feat_multi.index.dayofyear
df_feat_multi["Season_Sin"] = np.sin(2 * np.pi * df_feat_multi["DayOfYear"] / 365.25)
df_feat_multi["Season_Cos"] = np.cos(2 * np.pi * df_feat_multi["DayOfYear"] / 365.25)

# 4.1 Feature Engineering for all targets
N_LAGS = 3
ROLL_WINDOW = 3

for target in targets_config.keys():
    # Target-specific lags
    for lag in range(1, N_LAGS + 1):
        df_feat_multi[f"{target}_lag{lag}"] = df_feat_multi[target].shift(lag)

    # Target-specific rolling stats
    df_feat_multi[f"{target}_roll_mean3"] = (
        df_feat_multi[target].rolling(window=ROLL_WINDOW).mean()
    )
    df_feat_multi[f"{target}_roll_std3"] = (
        df_feat_multi[target].rolling(window=ROLL_WINDOW).std()
    )

# Save a full-index copy before dropping NaNs (for the full hand-off predictions)
df_feat_multi_full = df_feat_multi.copy()

base_calendar = ["Hour_Sin", "Hour_Cos", "Season_Sin", "Season_Cos"]

# We will store trained models and feature columns to plot later
trained_models = {}
feature_columns_by_target = {}

for target, info in targets_config.items():
    # Build feature list for this specific model
    feature_cols = (
        base_calendar
        + info["exog_cols"]
        + [
            f"{target}_lag1",
            f"{target}_lag2",
            f"{target}_lag3",
            f"{target}_roll_mean3",
            f"{target}_roll_std3",
        ]
    )
    feature_columns_by_target[target] = feature_cols

    # Drop rows containing NaNs in features/target for training/evaluation
    df_target = df_feat_multi.dropna(subset=feature_cols + [target]).copy()

    X_target = df_target[feature_cols]
    y_target = df_target[target]

    # Calculate Baselines
    # 1. Persistence baseline
    mse_pers = mean_squared_error(y_target, df_target[f"{target}_lag1"])

    # 2. Seasonal Naive baseline (same hour, 24 hours ago)
    seasonal_naive = df_feat_multi[target].shift(24).reindex(df_target.index)
    mask = seasonal_naive.notna()
    mse_seas = mean_squared_error(y_target[mask], seasonal_naive[mask])

    # Use a standard 20% test split with TimeSeriesSplit to avoid data leakage (no shuffling, respects temporal order)
    tscv = TimeSeriesSplit(n_splits=5)
    train_idx, test_idx = list(tscv.split(X_target))[-1]
    X_train, X_test = X_target.iloc[train_idx], X_target.iloc[test_idx]
    y_train, y_test = y_target.iloc[train_idx], y_target.iloc[test_idx]

    # Train the Random Forest
    model = RandomForestRegressor(max_depth=5, n_estimators=20, random_state=42)
    model.fit(X_train, y_train)
    trained_models[target] = model

    # Predict and evaluate
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Apply physical clipping if specified
    if info["clip_zero"]:
        y_pred_train = np.maximum(0, y_pred_train)
        y_pred_test = np.maximum(0, y_pred_test)

    mse_train = mean_squared_error(y_train, y_pred_train)
    mse_test = mean_squared_error(y_test, y_pred_test)

    variance_test = y_test.var(ddof=0)
    nmse_test = mse_test / variance_test

    print(f"\n" + "=" * 50)
    print(f"🎯 Target Model: {target}")
    print(f"=" * 50)
    print(f"📊 Training MSE: {mse_train:.6f}")
    print(f"🔮 Test MSE:     {mse_test:.6f}")
    print(f"📉 Test NMSE:     {nmse_test:.4f} (R² equivalent: {1 - nmse_test:.4f})")
    print(f"--- Baselines for comparison ---")
    print(f"Persistence MSE:    {mse_pers:.6f}")
    print(f"Seasonal Naive MSE: {mse_seas:.6f}")


# Save each model to a file
for target, info in targets_config.items():
    joblib.dump(trained_models[target], f"rf_{target}_model.joblib")

print("All models successfully saved as .joblib files.")



🎯 Target Model: Wind_kW
📊 Training MSE: 0.000173
🔮 Test MSE:     0.040184
📉 Test NMSE:     0.0610 (R² equivalent: 0.9390)
--- Baselines for comparison ---
Persistence MSE:    0.070043
Seasonal Naive MSE: 0.402589

🎯 Target Model: Demand_kW
📊 Training MSE: 1.422799
🔮 Test MSE:     2.295424
📉 Test NMSE:     0.0509 (R² equivalent: 0.9491)
--- Baselines for comparison ---
Persistence MSE:    19.300045
Seasonal Naive MSE: 8.376531

🎯 Target Model: Price_EUR_per_kWh
📊 Training MSE: 0.000046
🔮 Test MSE:     0.000084
📉 Test NMSE:     0.0230 (R² equivalent: 0.9770)
--- Baselines for comparison ---
Persistence MSE:    0.002630
Seasonal Naive MSE: 0.000212
All models successfully saved as .joblib files.
